# Mobile-Money Agent Network — Transaction Cleaning & Validation

This notebook loads the raw agent and transaction datasets, preserves a raw backup (`raw_backup`), cleans transaction fields and flags anomalies, joins the agent register safely, saves `outputs/agent_transactions_clean.csv`, and documents the cleaning log and operational recommendations.

## Setup, Raw Backup & Initial Inspection

In [ ]:
import os
import numpy as np
import pandas as pd
from cleaning_pipeline import clean_agent_data

agents_raw = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv")
raw_backup = txns_raw.copy(deep=True)

print("Raw agents shape:", agents_raw.shape)
print("Raw transactions shape:", txns_raw.shape)
display(agents_raw.head())
display(txns_raw.head(10))
print("\nRaw txn_type counts (including NaN):")
print(txns_raw["txn_type"].value_counts(dropna=False))
print("\nRaw missing values per column:")
print(txns_raw.isna().sum())

## Step-by-Step Cleaning via `clean_agent_data(txns, agents)`

In [ ]:
clean_df, cleaning_log = clean_agent_data(txns_raw, agents_raw)

print("Cleaning Log:")
for item in cleaning_log:
    print("-", item)

print("\nCleaned DataFrame shape:", clean_df.shape)
display(clean_df.head(10))

## Join Comparison (Inner vs. Left vs. Outer) & Validation

In [ ]:
dedup_txns = txns_raw.drop_duplicates().copy()
dedup_txns["amount_num"] = pd.to_numeric(
    dedup_txns["amount"].astype("string").str.strip().str.replace(",", "", regex=False),
    errors="coerce",
)
dedup_txns["amount_for_loss_calc"] = dedup_txns["amount_num"].fillna(dedup_txns["amount_num"].median())

inner_join = dedup_txns.merge(agents_raw, on="agent_id", how="inner", validate="many_to_one")
left_join = dedup_txns.merge(agents_raw, on="agent_id", how="left", validate="many_to_one", indicator=True)
outer_join = dedup_txns.merge(agents_raw, on="agent_id", how="outer", validate="many_to_one")

print("Inner join rows:", len(inner_join))
print("Left join rows:", len(left_join))
print("Outer join rows:", len(outer_join))

unmatched_txns = left_join[left_join["_merge"] == "left_only"]
print("\nUnmatched transactions (e.g. AG099) count:", len(unmatched_txns))
print("Raw numeric sum of unmatched transactions:", unmatched_txns["amount_num"].sum())
print("Imputed-adjusted sum of unmatched transactions:", unmatched_txns["amount_for_loss_calc"].sum())
display(unmatched_txns[["txn_id", "agent_id", "txn_type", "amount", "txn_date", "_merge"]].head())

## Export Cleaned Dataset (`outputs/agent_transactions_clean.csv`)

In [ ]:
os.makedirs("outputs", exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved outputs/agent_transactions_clean.csv with shape:", clean_df.shape)

## Cleaning Log & Operational Recommendations
- **Duplicate rows:** Exact duplicate transaction rows were removed prior to aggregation so transactions are not double-counted.
- **Transaction types & dates:** `txn_type` values were trimmed, lowercased, and mapped cleanly to `Deposit` and `Withdrawal` (with any unmapped entries marked `Unknown`), while `txn_date` values with mixed `/` and `-` separators were standardised to `YYYY-MM-DD`.
- **Missing/invalid amounts & outliers:** Non-numeric (`"N/A"`) and missing amounts were flagged in `amount_imputed` before median imputation, and high-value transactions were flagged in `is_outlier` using the IQR rule rather than silently deleted.
- **Unregistered agent IDs (`AG099`):** Using a left join instead of an inner join retains transactions tied to `AG099` with `county` set to `Unknown`, preventing silent revenue/volume loss and giving operations a clear queue to investigate unmapped agent codes.